# A Quick Recap for the each part of the AI agents field


##  LangChain vs LangGraph State & Component Comparison

This document summarizes the key concepts discussed: **LangChain**, **LangGraph**, **AgentState**, **TypedDict State**, **dataclass Context**, and **Pydantic BaseModel**.

---

### 1. High-Level Architecture

| Layer | Purpose | Typical Components | Who Owns It |
|------|---------|-------------------|-------------|
| Capability Layer | Provides AI functionality | LLMs, Tools, Prompts | LangChain |
| Execution Layer | Controls flow of execution | Graph nodes, edges, reducers | LangGraph |
| Agent Memory | Stores agent reasoning state | AgentState | LangChain |
| Workflow State | Data passed between steps | TypedDict State | LangGraph |
| Runtime Context | External configuration | dataclass context | Application |
| Data Contracts | Validate structured input/output | Pydantic BaseModel | Tools / APIs |

---

### 2. LangChain vs LangGraph

| Feature | LangChain | LangGraph |
|-------|-----------|-----------|
| Primary Role | AI capability framework | Execution orchestration |
| Core Concept | Agents | Graph workflows |
| Control Flow | Implicit agent loop | Explicit graph |
| State Management | Agent memory | Workflow state |
| Parallel Execution | Limited | Native |
| Determinism | Lower | High |
| Production Workflows | Harder to manage | Designed for production |
| Tooling | LLMs, prompts, tools | Nodes, edges, reducers |
| Interrupt Support | Agent-level | Graph-level |
| Checkpointing | Limited | Built-in |

---

### 3. AgentState vs TypedDict State

| Aspect | AgentState | TypedDict State |
|------|-------------|----------------|
| Library | LangChain | LangGraph |
| Purpose | Agent internal memory | Workflow shared state |
| Structure | Python class | Dictionary schema |
| Mutation Style | Mutable object | Functional updates |
| Scope | Single agent | Entire workflow |
| Supports Reducers | No | Yes |
| Parallel Updates | No | Yes |
| Context Growth | Continuous | Controlled |
| Used By | Tools, middleware | Graph nodes |
| Control Flow | Agent loop | Graph edges |

---

### 4. dataclass Context vs AgentState

| Aspect | dataclass Context | AgentState |
|------|------------------|-----------|
| Purpose | Runtime configuration | Agent memory |
| Mutability | Typically static | Mutable |
| Lifecycle | Provided at invocation | Changes during execution |
| Storage | Outside the agent | Inside the agent |
| Example Data | API keys, user info | authentication flags |
| Persistence | No | Optional |
| Context Window Impact | None | Yes |


### 5. TypedDict vs BaseModel

| Aspect | TypedDict | BaseModel |
|------|------------------|-----------|
| Library | Python typing | Pydantic |
| Purpose | Describe dictionary structure | Validate structured data |
| Runtime Validation | No | Yes |
| Used In | Graph state |Tools / APIs |
| Serialization | Manual | Automatic |
| Strict Typing | Static only | Runtime enforcedOptional |
| Ideal For | Workflow data flow | API schemas  |



### 6. State Evolution Comparison

| Property        | AgentState          | TypedDict         |
| --------------- | ------------------- | ----------------- |
| Memory Pattern  | Accumulating memory | Selective updates |
| Context Size    | Grows continuously  | Controlled        |
| Data Ownership  | Agent               | Workflow          |
| Update Method   | Attribute mutation  | Return dictionary |
| Parallel Safety | No                  | Yes               |


### 7. Mental Model Summary

| Concept           | Think of it as                       |
| ----------------- | ------------------------------------ |
| LangChain         | AI capability toolkit                |
| LangGraph         | Workflow execution engine            |
| AgentState        | What the agent remembers             |
| TypedDict State   | Data flowing through the workflow    |
| dataclass Context | Configuration provided to the system |
| BaseModel         | Contract enforcing structured data   |


### 8. Final Simplified Architecture Diagram
```text
Application
   |
   |-- Context (dataclass)
   |
LangGraph Workflow
   |
   |-- State (TypedDict)
   |
   |-- Node
        |
        |-- LangChain Agent
                |
                |-- AgentState
                |-- Tools
                |-- LLM
```

# Langchain Docs, References: 

[Langchain Docs](https://docs.langchain.com/)

[Langchain Python Reference](https://reference.langchain.com/python/)

![alt text](<../../assets/Full Rag Pipeline.png>)

### Environment Initialization

Loads environment variables from a `.env` file and configures:
- **LangSmith tracing** (`LANGCHAIN_TRACING_V2`, `LANGCHAIN_API_KEY`, `LANGCHAIN_PROJECT`, `LANGCHAIN_ENDPOINT`) — enables observability and debugging of LangChain runs.
- **Mistral API key** — authenticates calls to the Mistral LLM and embedding models.
- **HuggingFace token** — for any HF model access.
- **User-Agent header** — required by `WebBaseLoader` to fetch web pages.

Warnings are suppressed for cleaner output.

In [ ]:
import warnings
import os 
from dotenv import load_dotenv

# Suppress all warnings for cleaner notebook output
warnings.filterwarnings("ignore")

# Load environment variables from the .env file into the process
load_dotenv()

try: 
    # Map .env variables to the keys LangChain/LangSmith expects at runtime
    os.environ["LANGCHAIN_TRACING_V2"] = os.getenv("LANGSMITH_TRACING_V2")   # Enable LangSmith tracing
    os.environ["LANGCHAIN_API_KEY"] = os.getenv("LANGSMITH_API_KEY")         # LangSmith authentication key
    os.environ["LANGCHAIN_PROJECT"] = os.getenv("LANGSMITH_PROJECT")         # LangSmith project name for grouping traces
    os.environ["LANGCHAIN_ENDPOINT"] = os.getenv("LANGSMITH_ENDPOINT")       # LangSmith API endpoint URL
    os.environ["MISTRAL_API_KEY"] = os.getenv("MISTRAL_API_KEY")             # Mistral AI LLM/embedding API key
    os.environ["HF_TOKEN"] = os.getenv("HF_TOKEN")                           # HuggingFace access token
    os.environ["USER_AGENT"] = "MyLangChainApp/1.0"                           # Required User-Agent header for WebBaseLoader
    print("Environment variables set successfully")
except Exception as e: 
    print(f"Error: {e}")

Environment variables set successfully


### Part 1: Full RAG Pipeline Overview

End-to-end RAG pipeline in a single cell covering all four stages:

1. **Load** — Fetches Lilian Weng's blog post on LLM agents using `WebBaseLoader` with `BeautifulSoup` filtering (`post-content`, `post-title`, `author`).
2. **Split** — Chunks the document with `RecursiveCharacterTextSplitter` (1000 chars, 200 overlap).
3. **Embed & Store** — Creates Mistral embeddings (`mistral-embed`) and persists them in a local ChromaDB collection (`"Tutorial"`).
4. **Retrieve & Generate** — Builds a LCEL chain: retriever → format docs → RAG prompt (pulled from LangSmith Hub `rlm/rag-prompt`) → `mistral-medium-latest` LLM → string output.

The chain is invoked with the question *"What is Task Decomposition?"*.

In [ ]:
import bs4
from langchain_text_splitters import RecursiveCharacterTextSplitter       # Text chunking utility
from langchain_community.document_loaders import WebBaseLoader            # Fetch web pages as Documents
from langchain_community.vectorstores import Chroma                       # ChromaDB vector store integration
from langchain_core.output_parsers import StrOutputParser                 # Extracts raw string from LLM response
from langchain_core.runnables import RunnablePassthrough                  # Passes input through unchanged in LCEL chains
from langchain_mistralai import ChatMistralAI, MistralAIEmbeddings        # Mistral LLM and embedding models
from langsmith import Client
client = Client()  # LangSmith client for pulling prompts and tracing


# ========================
# STAGE 1: LOAD DOCUMENTS
# ========================
# Fetch the blog post HTML, filtering only article-relevant CSS classes
loader = WebBaseLoader(
    web_paths = ["https://lilianweng.github.io/posts/2023-06-23-agent/"],
    bs_kwargs = dict(
        parse_only = bs4.SoupStrainer(
            class_ = ("post-content", "post-title", "author")  # Keep only main content
        )
    ),
)
docs = loader.load()  # Returns a list of Document objects

# ========================
# STAGE 2: SPLIT INTO CHUNKS
# ========================
# Break documents into overlapping chunks for better retrieval granularity
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 1000,    # Max 1000 characters per chunk
    chunk_overlap = 200,  # 200-char overlap to preserve context at boundaries
)
splits = text_splitter.split_documents(docs)

# ========================
# STAGE 3: EMBED & STORE
# ========================
# Create embeddings and persist them in a local ChromaDB collection
vectorstore = Chroma.from_documents(
    documents = splits,
    embedding = MistralAIEmbeddings(model = "mistral-embed"),  # Mistral embedding model
    collection_name = "Tutorial",       # Collection name inside ChromaDB
    persist_directory = "../db"          # Local directory for persistent storage
)

# ========================
# STAGE 4: RETRIEVE & GENERATE
# ========================
# Create a retriever from the vector store (default: top-4 results)
retriever = vectorstore.as_retriever()

# Pull the community RAG prompt template from LangSmith Hub
prompt = client.pull_prompt("rlm/rag-prompt")

# Initialize the LLM with deterministic output (temperature=0)
llm = ChatMistralAI(
    model = "mistral-medium-latest",
    temperature = 0,
)

# Helper: join retrieved document contents into a single context string
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

# Build the full LCEL RAG chain:
# 1. Retriever fetches docs → format_docs joins them as context
# 2. RunnablePassthrough carries the original question forward
# 3. Prompt → LLM → StrOutputParser produces the final text answer
rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

# Invoke the chain with a test question
rag_chain.invoke("what is Task Decomposition?")

'Task decomposition is the process of breaking down a complex task into smaller, manageable subgoals or steps. This can be done using simple prompts (e.g., "Steps for XYZ"), task-specific instructions, or human input. Another approach, like **LLM+P**, outsources planning to an external tool using structured languages like PDDL.'

### Part 2: Indexing — Embeddings & Similarity

This section demonstrates the **indexing** stage in detail:
- How text is tokenized and counted using `tiktoken` (`cl100k_base` encoding).
- How text is embedded into dense vectors using `MistralAIEmbeddings`.
- How **cosine similarity** measures semantic closeness between a question and a document.
- How documents are loaded, split (300 tokens, 50 overlap via tiktoken encoder), and stored in ChromaDB (`"blog_posts"` collection).

#### Define Sample Question & Document

Creates a simple question-document pair to demonstrate embedding and similarity. These will be tokenized, embedded, and compared in the following cells.

In [ ]:
# Sample texts to demonstrate tokenization, embedding, and similarity comparison
question = "What kinds of pets do I like?"
document =  "My favorite pet is cat."

#### Token Counting with Tiktoken

Uses the `tiktoken` library (`cl100k_base` encoding, same as OpenAI models) to count the number of tokens in a string. This helps understand how text length maps to token consumption and embedding costs.

In [ ]:
import tiktoken

def num_tokens_from_string(string: str, encoding_name: str) -> int:
    """Return the number of tokens in a text string."""
    # Load the tokenizer encoding (cl100k_base is used by GPT-4 / text-embedding-ada-002)
    encoding = tiktoken.get_encoding(encoding_name)
    # Encode the string and count the resulting tokens
    num_tokens = len(encoding.encode(string))
    return num_tokens

# Count tokens in the sample question using the cl100k_base encoding
num_of_tokens = num_tokens_from_string(question, "cl100k_base")

num_of_tokens

8

#### Generate Embeddings

Embeds both the question and the document into dense vectors using `MistralAIEmbeddings` (`mistral-embed`). Returns the embedding dimension (vector length) to verify the model output.

In [ ]:
# Initialize the Mistral embedding model
embed = MistralAIEmbeddings(model = "mistral-embed")

# Generate dense vector representations for the question and document
query_result = embed.embed_query(question)    # Embed the question string
doc_result = embed.embed_query(document)      # Embed the document string

# Check the dimensionality of the embedding vector
len(query_result)

1024

#### Cosine Similarity

Computes cosine similarity between the question and document embeddings using NumPy. A value close to 1.0 means the texts are semantically similar. This is the same metric used internally by vector stores for retrieval.

In [ ]:
import numpy as np 

def cosine_similarity(vec1, vec2):
    """Compute cosine similarity between two vectors.
    Returns a value in [-1, 1]; closer to 1 = more similar."""
    dot_product = np.dot(vec1, vec2)        # Numerator: dot product of the two vectors
    norm_vec1 = np.linalg.norm(vec1)        # L2 norm of vector 1
    norm_vec2 = np.linalg.norm(vec2)        # L2 norm of vector 2
    return dot_product / (norm_vec1 * norm_vec2)  # Cosine similarity formula

# Compare the question and document embeddings
similarity_result = cosine_similarity(query_result, doc_result)

print(similarity_result)  # High value indicates semantic similarity

0.7558983474652954


#### Load Blog Document

Fetches Lilian Weng's blog post using `WebBaseLoader` with BeautifulSoup filtering for `post-content`, `post-title`, and `author` classes. This isolates the main article content from navigation, ads, and other page elements.

In [ ]:
### Indexing ###

# Load the blog post, filtering for article content only via BeautifulSoup
loader = WebBaseLoader(
    web_paths = ["https://lilianweng.github.io/posts/2023-06-23-agent/"],
    bs_kwargs = dict(
        parse_only = bs4.SoupStrainer(
            class_ = ("post-content", "post-title", "author")  # Filter to main content classes
        )
    ),
)

blog_docs = loader.load()  # Returns list of Document objects with page_content and metadata

#### Split Documents with Tiktoken Encoder

Splits the loaded blog into chunks using `RecursiveCharacterTextSplitter.from_tiktoken_encoder` with 300-token chunks and 50-token overlap. Using tiktoken ensures chunk sizes align with token counts rather than character counts.

In [ ]:
# Create a splitter calibrated to token counts (not character counts)
text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    chunk_size = 300,     # Max 300 tokens per chunk
    chunk_overlap = 50,   # 50-token overlap to maintain context at chunk boundaries
)

# Split the loaded blog documents into chunks
splits = text_splitter.split_documents(blog_docs)

#### Store Embeddings in ChromaDB

Creates a persistent ChromaDB vector store (`"blog_posts"` collection) from the split documents using Mistral embeddings. The `persist_directory` ensures data survives across sessions. A retriever is created from the store for downstream use.

In [ ]:
# ========================
# Vector Store — Persist Embeddings in ChromaDB
# ========================

# Initialize Mistral embedding model for vectorization
embeddings = MistralAIEmbeddings(model = "mistral-embed")

# Create a ChromaDB collection from the document chunks with persistent storage
vStore = Chroma.from_documents(
    splits,                              # Chunked documents from the splitter
    embedding=embeddings,                # Embedding function to vectorize each chunk
    collection_name="blog_posts",        # Name of the ChromaDB collection
    persist_directory="../db_blog",      # Directory to persist the database on disk
)

# Wrap the vector store as a LangChain retriever for downstream chain use
retriever = vStore.as_retriever()

### Part 3: Retrieval

Demonstrates loading an **existing** ChromaDB vector store and querying it with a retriever.

- Connects to the persisted `"blog_posts"` collection using the same embedding function.
- Creates a retriever with `search_kwargs={"k": 1}` to return only the single most relevant chunk.
- Invokes the retriever with a sample question and returns the number of matching documents.

In [ ]:
# Re-initialize embedding model (same model used during indexing)
embeddings = MistralAIEmbeddings(model = "mistral-embed")

# Load the previously persisted ChromaDB vector store from disk
vStore = Chroma(
    persist_directory="../db_blog",      # Path to the persisted database
    embedding_function=embeddings,       # Must match the embedding function used at creation time
    collection_name="blog_posts",        # Target collection within the database
)

# Create a retriever that returns only the top-1 most relevant chunk
retriever = vStore.as_retriever(search_kwargs={"k": 1})

# Test retrieval with a sample question
docs = retriever.invoke("what is Task Decomposition?")
len(docs)  # Should return 1 (matching k=1)

1

### Part 4: Generation

Builds the **generation** side of RAG using an LLM and prompt templates.

Two approaches are shown:
1. **Manual prompt template** — A `ChatPromptTemplate` with `{context}` and `{question}` placeholders, piped through `mistral-small-latest` and `StrOutputParser`.
2. **LangSmith Hub prompt** — Pulls the community `rlm/rag-prompt` template and wires it into a full LCEL RAG chain: retriever → format docs → prompt → LLM → string output.

In [ ]:
from langchain_mistralai import ChatMistralAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# Define a RAG prompt template with two placeholders:
#   {context}  — retrieved document content injected here
#   {question} — the user's original question
template = """Answer the question based only on the following context: 
{context}

Question: {question}
"""

# Build a ChatPromptTemplate from the string template
prompt = ChatPromptTemplate.from_template(template)
prompt

ChatPromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, template='Answer the question based only on the following context: \n{context}\n\nQuestion: {question}\n'), additional_kwargs={})])

#### Manual Prompt Template & Chain

Defines a `ChatPromptTemplate` with context and question slots. Chains it with `mistral-small-latest` (temperature=0) and `StrOutputParser` to produce a deterministic text answer from the full blog documents.

In [ ]:
# Initialize the LLM with deterministic output (temperature=0)
llm = ChatMistralAI(model = "mistral-small-latest", temperature = 0)

# Build a simple LCEL chain: prompt → LLM → string output parser
chain = prompt | llm | StrOutputParser()

# Run the chain, passing the full blog docs as context (no retriever here)
result = chain.invoke({"context": blog_docs, "question": "What is Task Decomposition?"})

print(result)

Task decomposition is the process of breaking down complex tasks into smaller, manageable subgoals or steps. It is often achieved through techniques like Chain of Thought (CoT) or Tree of Thoughts (ToT), which help decompose tasks into simpler, sequential steps. This approach enhances problem-solving by making large tasks more manageable and interpretable.


#### Full RAG Chain with LangSmith Hub Prompt

Pulls the `rlm/rag-prompt` from LangSmith Hub and assembles the complete RAG chain:
- **Retriever** fetches relevant chunks → `format_docs` joins them into a single string.
- **`RunnablePassthrough`** carries the original question forward.
- The prompt, LLM, and output parser produce the final answer.

This is the canonical LCEL pattern for RAG.

In [ ]:
# Pull the community RAG prompt template from LangSmith Hub
from langsmith import Client
client = Client()
prompt_temp = client.pull_prompt("rlm/rag-prompt")

# Build the full LCEL RAG chain:
#   1. "context": retriever fetches relevant chunks → format_docs joins them
#   2. "question": RunnablePassthrough carries the user query through unchanged
#   3. prompt_temp → llm → StrOutputParser produces the final text answer
rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()} 
    | prompt_temp
    | llm
    | StrOutputParser()
)

# Invoke the chain with a test question and print the answer
print(rag_chain.invoke("What is Task Decomposition?"))

Task decomposition is the process of breaking down complex tasks into smaller, more manageable steps. It helps agents plan and execute tasks efficiently by simplifying the problem-solving process. This can be done through prompting, task-specific instructions, or human input.
